In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from jax import numpy as jnp
from vizopt.radially_convex import optimize_multiple_radially_convex_sets_with_movable_circles
from vizopt.animation import SnapshotCallback

In [ ]:
rng = np.random.default_rng(0)

n_sets = 3
n_circles_per_set = 5


In [ ]:
N = n_sets * n_circles_per_set
positions = rng.uniform(-3.0, 3.0, size=(N, 2)).astype(np.float32)
radii = rng.uniform(0.3, 0.7, size=(N, 1)).astype(np.float32)
circles = np.hstack([positions, radii])  # shape (N, 3): [cx, cy, r]

sets = [
    list(range(s * n_circles_per_set, (s + 1) * n_circles_per_set))
    for s in range(n_sets)
]

In [ ]:
# ── Optimize ──────────────────────────────────────────────────────────────────
snapshot_cb = SnapshotCallback(every=200)

def warmup_schedule(step):
    return jnp.minimum(1.0, step / 3000)

def late_warmup_schedule(step):
    return jnp.clip((step - 3000)/3000, min=0.01, max=1.0)

term_schedules = {"exclusion": warmup_schedule,
                  "area": late_warmup_schedule,
                  "perimeter": late_warmup_schedule,
                  "circle_collision": warmup_schedule}

results, circles_out, history = optimize_multiple_radially_convex_sets_with_movable_circles(
    circles,
    sets,
    k_angles=64,
    weight_area=1.0,
    weight_perimeter=2.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.1,
    weight_circle_collision=50.0,
    weight_bounding_box=1.,
    optim_kwargs={"n_iters": 5000, "learning_rate": 0.005, "callback": snapshot_cb},
    #term_schedules=term_schedules
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
import matplotlib.cm as cm
import matplotlib.animation as manimation
from matplotlib import pyplot as plt
from IPython.display import HTML

circle_radii = np.array([c[2] for c in circles])
angles_anim = np.linspace(0, 2 * np.pi, 64, endpoint=False)
cmap_anim = cm.get_cmap("tab20", len(sets))

# Fixed axis limits: use a bounding box from final result plus margin
all_x, all_y = [], []
for result in results:
    cx, cy = result["center"]
    all_x.extend((cx + result["radii"] * np.cos(result["angles"])).tolist())
    all_y.extend((cy + result["radii"] * np.sin(result["angles"])).tolist())
margin = 0.5
xlim = (min(all_x) - margin, max(all_x) + margin)
ylim = (min(all_y) - margin, max(all_y) + margin)

order = sorted(range(len(sets)), key=lambda s: len(sets[s]), reverse=True)

def render_snapshot(optim_vars):
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")

    for s in order:
        cx, cy = optim_vars["centers"][s]
        s_radii = optim_vars["radii"][s]
        bx = np.append(cx + s_radii * np.cos(angles_anim), cx + s_radii[0] * np.cos(angles_anim[0]))
        by = np.append(cy + s_radii * np.sin(angles_anim), cy + s_radii[0] * np.sin(angles_anim[0]))
        color = cmap_anim(s)
        ax.fill(bx, by, alpha=0.10, color=color)
        ax.plot(bx, by, color=color, linewidth=1.0, alpha=0.7)

    # Draw circles at their current optimized positions
    circle_positions = optim_vars["circle_positions"]  # shape (N, 2)
    for s in order:
        color = cmap_anim(s)
        for i in sets[s]:
            cx_c, cy_c = circle_positions[i]
            r = circle_radii[i]
            patch = plt.Circle((cx_c, cy_c), r, color=color, alpha=0.4, linewidth=0.5, edgecolor=color)
            ax.add_patch(patch)

    return fig

# Build frames
images = []
for i_iter, optim_vars in snapshot_cb.snapshots:
    fig = render_snapshot(optim_vars)
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).copy()
    images.append((i_iter, buf.reshape(h, w, 4)))
    plt.close(fig)

# Assemble animation
fig_anim, ax_anim = plt.subplots()
ax_anim.axis("off")
fig_anim.tight_layout(pad=0)
im = ax_anim.imshow(images[0][1])
title = ax_anim.set_title(f"Iteration {images[0][0]}")

def update(frame):
    i_iter, img = images[frame]
    im.set_data(img)
    title.set_text(f"Iteration {i_iter}")
    return [im, title]

anim = manimation.FuncAnimation(fig_anim, update, frames=len(images), interval=150, blit=True)
HTML(anim.to_jshtml())